# Generate Diagnostics for Higher-Dimensional Inputs

This Generate notebook computes and caches the data consumed by `Visualize-higher-dimensional-layer-averaged-diagnostics-across-context-lengths.ipynb`. The companion Visualize notebook shows layer-averaged graph-spectral diagnostics across context lengths for three-dimensional inputs using Llama-3.2-3B, as presented under **Additional Controls and Higher-Dimensional Inputs** in the appendix.

This extension tests whether the context- and complexity-dependent trends observed for one-dimensional inputs extend to higher dimensions. Inputs comprise constant vectors, three independent logistic-map sequences sharing the same parameter $r$, and coupled Lorenz $(x,y,z)$ trajectories. Commas separate coordinates within each state; semicolons separate time steps.


Set imports, constants, output paths, sweep axes, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import hashlib
import json
import random
from pathlib import Path
import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model and sweep configuration
SEED = 42
ENSEMBLE_SEED = SEED
MODEL_NAME = "meta-llama/Llama-3.2-3B"
EXPECTED_NUM_LAYERS = 28
NUM_ENSEMBLE_RUNS = 20
INPUT_SUITE = "3-channel-numerical"
if INPUT_SUITE != "3-channel-numerical":
    raise ValueError("Use the companion original-setup notebook for scalar inputs.")
NUM_CHANNELS = 3 if INPUT_SUITE == "3-channel-numerical" else 1
TOKENS_PER_TIME_STEP = 2 * NUM_CHANNELS
STATE_DELIMITER = ";" if NUM_CHANNELS > 1 else ","
EXPECTED_NUM_INPUT_FAMILIES = 10
# Independent per-channel min/max over the full retained trajectory; never
# pool channels or rescale context prefixes. Match the 1D round(50 + 900*z)
QUANTIZATION_CONVENTION = "per_channel_full_retained_trajectory_minmax_050_950"
# Sampled constants bypass trajectory min/max, just as in the original suite
CONSTANT_QUANTIZATION_CONVENTION = "per_channel_round_50_plus_900_times_uniform_0.3_0.7"

# Diagnostics computed at each context length
# Keep the original token budgets, rounding to complete states (N = 6T)
TARGET_CONTEXT_TOKEN_COUNTS = np.arange(100, 2001, 100, dtype=np.int64)
REQUESTED_CONTEXT_STATE_COUNTS = np.rint(
    TARGET_CONTEXT_TOKEN_COUNTS / TOKENS_PER_TIME_STEP
).astype(np.int64)
MAX_CONTEXT_STATES = int(REQUESTED_CONTEXT_STATE_COUNTS[-1])
TOTAL_SEQUENCE_STATES = max(1000, MAX_CONTEXT_STATES)
LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES = 10
LOW_SPECTRUM_LAMBDA_INDICES = np.arange(
    2,
    2 + LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES,
    dtype=np.int64,
)
HFER_TAIL_FRACTIONS = np.asarray([0.50, 0.40, 0.30, 0.20], dtype=np.float64)
HFER_TAIL_LABELS = np.asarray(
    ["top_50_percent", "top_40_percent", "top_30_percent", "top_20_percent"],
    dtype=str,
)

# Checkpoint progress; save final plot data to OUTPUT_PATH
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_TASKS = 20

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_STEM = (
    "three_channel_numerical_diagnostics_averaged_over_layers_vs_context_length"
    if NUM_CHANNELS > 1 else "all_diagnostics_averaged_over_layers_vs_context_length"
)
OUTPUT_PATH = CACHE_DIR / f"{OUTPUT_STEM}.npz"
CHECKPOINT_PATH = CACHE_DIR / f"{OUTPUT_STEM}.resume_checkpoint.npz"

NPZ_SCHEMA_VERSION = "layer_averaged_vs_context_length_multichannel_v2"
CHECKPOINT_SCHEMA_VERSION = NPZ_SCHEMA_VERSION + "_checkpoint"

DROP_BOS = True
RENORMALIZE_AFTER_DROP = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
HEAD_WEIGHTING = "mass"
TOKENIZATION_ADD_SPECIAL_TOKENS = True
K_CUTOFF = None

# Sweep axes
context_fracs_full = REQUESTED_CONTEXT_STATE_COUNTS / MAX_CONTEXT_STATES
context_percent_full = np.asarray([int(round(100 * frac)) for frac in context_fracs_full], dtype=np.int64)
run_ids_full = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)

input_data_order_full = np.asarray([
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
], dtype=str)

if NUM_CHANNELS > 1:
    input_data_order_full = np.asarray([
        name.replace("Logistic", "3-channel logistic", 1)
        for name in input_data_order_full
    ], dtype=str)

# Run the full sweep
active_input_indices = np.arange(input_data_order_full.size, dtype=np.int64)
active_run_indices = np.arange(run_ids_full.size, dtype=np.int64)
active_context_indices = np.arange(context_fracs_full.size, dtype=np.int64)

input_names = input_data_order_full[active_input_indices]
run_ids = run_ids_full[active_run_indices]
context_fracs = context_fracs_full[active_context_indices]
context_percent = context_percent_full[active_context_indices]

diagnostic_names = np.asarray([
    "layer_averaged_effective_active_modes",
    "layer_averaged_normalized_fiedler_value_lambda_2",
    "layer_averaged_hfer",
    "layer_averaged_low_end_nontrivial_normalized_laplacian_eigenvalues",
    "layer_averaged_hfer_high_frequency_tail_energy_by_fraction",
], dtype=str)

# Seed data generation and model inference
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Define sequence generators and labels for the input families.

In [ ]:
# Periodic and chaotic logistic-map regimes
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_expected_periods = {
    r: periodic_logistic_periods.get(r)
    for r in list(periodic_logistic_periods.keys()) + [3.9, 4.0]
}
logistic_long_n = 20000
logistic_settle_tol = 1e-10
periodic_random_phase = True

# Match Lorenz sampling to the sequence length
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
LORENZ_BASE_INITIAL_STATE = np.asarray([-8.0, 7.0, 27.0])
lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES, dtype=float) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))


def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


def logistic_sequence(r, x0, n):
    x = np.empty(int(n), dtype=float)
    x[0] = float(x0)
    for i in range(1, int(n)):
        x[i] = float(r) * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    if vals.ndim not in (1, 2) or vals.shape[0] == 0 or not np.isfinite(vals).all():
        raise ValueError("Expected a finite (time,) or (time, channel) trajectory.")
    # axis=0 scales each channel independently over the FULL retained window
    # This exactly retains the 1D convention; constant channels map to 500
    vmin, vmax = vals.min(axis=0), vals.max(axis=0)
    span = vmax - vmin
    scaled = np.round(50 + (vals - vmin) / np.where(span == 0, 1, span) * 900)
    return np.where(span == 0, 500, scaled).astype(int)


def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    # Use settled periodic windows with an optional phase shift
    if expected_period is None:
        vals = logistic_sequence(r, x0, n_window)
        return np.asarray(vals, dtype=float), 0, 0, False

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    vals = np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)
    return vals, int(t_settle), int(phase_offset), True


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


def _display_name(data_name):
    # Cache labels for visualization
    if NUM_CHANNELS > 1 and str(data_name) == "Constant sampled":
        return "3-channel constant sampled"
    if NUM_CHANNELS > 1 and str(data_name) == "Lorenz":
        return "Lorenz (x, y, z)"
    if str(data_name).startswith("3-channel logistic"):
        return _display_name(str(data_name).replace("3-channel logistic", "Logistic", 1)).replace(
            "Logistic", "3-channel logistic", 1
        )
    display_names = {
        "Constant sampled": "Constant sampled",
        "Logistic r=3.2": "Logistic r=3.2 (period 2)",
        "Logistic r=3.5": "Logistic r=3.5 (period 4)",
        "Logistic r=3.55": "Logistic r=3.55 (period 8)",
        "Logistic r=3.566": "Logistic r=3.566 (period 16)",
        "Logistic r=3.5692": "Logistic r=3.5692 (period 32)",
        "Logistic r=3.5698": "Logistic r=3.5698 (period 64)",
        "Logistic r=3.9": "Logistic r=3.9 (chaotic)",
        "Logistic r=4.0": "Logistic r=4.0 (chaotic)",
        "Lorenz": "Lorenz",
    }
    return display_names.get(str(data_name), str(data_name))


def _input_class_label(data_name):
    data_name = str(data_name).replace("3-channel logistic", "Logistic", 1)
    if data_name == "Constant sampled":
        return "constant"
    if data_name in {f"Logistic r={r}" for r in periodic_logistic_periods}:
        return "periodic"
    if data_name in {"Logistic r=3.9", "Logistic r=4.0"}:
        return "chaotic"
    if data_name == "Lorenz":
        return "lorenz"
    raise ValueError(f"Unknown input family: {data_name}")


input_labels = np.asarray([_display_name(name) for name in input_names], dtype=str)
input_class_labels = np.asarray([_input_class_label(name) for name in input_names], dtype=str)

if input_names.shape != (EXPECTED_NUM_INPUT_FAMILIES,):
    raise ValueError(f"Expected {EXPECTED_NUM_INPUT_FAMILIES} inputs, got {input_names.shape}.")
if run_ids.shape != (NUM_ENSEMBLE_RUNS,):
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} run ids, got {run_ids.shape}.")
if context_fracs.shape != (20,):
    raise ValueError(f"Expected 20 context fractions, got {context_fracs.shape}.")



Build ten input families from the same realization-level random samples: sampled constant vectors, independent logistic channels, and coupled Lorenz coordinates. Periodic logistic channels settle and randomize phase separately; chaotic logistic and Lorenz trajectories retain the original no-burn-in behavior.

In [ ]:
# Share each replicate's x0 across input families
ensemble_rng = np.random.default_rng(ENSEMBLE_SEED)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=(NUM_ENSEMBLE_RUNS, NUM_CHANNELS))
phase_rng = np.random.default_rng(ENSEMBLE_SEED + 1)

ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
ensemble_chaotic_rs = [3.9, 4.0]

constant_tokens = np.full((NUM_ENSEMBLE_RUNS, NUM_CHANNELS), -1, dtype=np.int64)
ensemble_inputs = {}

for rep_id, x0_channels in enumerate(x0_samples):
    rep_id = int(rep_id)
    x0_rep = float(x0_channels[0])
    rep_inputs = {}

    # Each channel is constant in time; preserve sampled values rather than
    # mapping these degenerate trajectories to the min/max fallback of 500
    constant_vector = np.round(50 + 900 * x0_channels).astype(np.int64)
    constant_tokens[rep_id] = constant_vector
    constant_sequence = np.tile(constant_vector, (TOTAL_SEQUENCE_STATES, 1))
    rep_inputs["Constant sampled"] = {
        "sequence": constant_sequence if NUM_CHANNELS > 1 else constant_sequence[:, 0],
        "metadata": {
            "input_class": "constant", "rep_id": rep_id,
            "initial_samples": x0_channels.tolist(),
            "constant_tokens": constant_vector.tolist(),
            "quantization_convention": CONSTANT_QUANTIZATION_CONVENTION,
        },
    }

    # Generalize the original scalar generator; all channels share r
    for r in ensemble_periodic_rs + ensemble_chaotic_rs:
        expected_period = logistic_expected_periods[r]
        channel_windows, channel_metadata = [], []
        for x0_channel in x0_channels:
            phase_offset = (
                int(phase_rng.integers(0, expected_period))
                if expected_period is not None and periodic_random_phase else 0
            )
            vals, burn_in_start_index, used_phase_offset, used_burn_in = (
                build_burned_in_logistic_window_ensemble(
                    r, x0=float(x0_channel), n_window=TOTAL_SEQUENCE_STATES,
                    expected_period=expected_period, n_long=logistic_long_n,
                    tol=logistic_settle_tol, phase_offset=phase_offset,
                )
            )
            channel_windows.append(vals)
            channel_metadata.append({
                "x0": float(x0_channel),
                "burn_in_start_index": int(burn_in_start_index),
                "phase_offset": int(used_phase_offset),
                "used_burn_in": bool(used_burn_in),
                "quantization_min": float(vals.min()),
                "quantization_max": float(vals.max()),
            })
        values = np.column_stack(channel_windows)
        sequence = quantize_logistic_window(values)
        data_name = f"3-channel logistic r={r}" if NUM_CHANNELS > 1 else f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": sequence if NUM_CHANNELS > 1 else sequence[:, 0],
            "metadata": {
                "input_class": "periodic" if expected_period is not None else "chaotic",
                "rep_id": rep_id, "r": float(r), "expected_period": expected_period,
                "channels": channel_metadata,
            },
        }

    # Extend the original x0 = -8 + (u - 0.5) rule independently to y0,z0
    # The three coordinates then evolve TOGETHER in one coupled Lorenz solve
    # For the legacy scalar suite, only x0 is perturbed and only x is observed
    lorenz_perturbation = np.zeros(3, dtype=float)
    lorenz_perturbation[:NUM_CHANNELS] = x0_channels - 0.5
    lorenz_initial_state = LORENZ_BASE_INITIAL_STATE + lorenz_perturbation
    sol_rep = solve_ivp(
        lorenz, t_span, lorenz_initial_state,
        args=(sigma, rho, beta), t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape != (3, TOTAL_SEQUENCE_STATES):
        raise ValueError(f"Unexpected Lorenz trajectory shape: {sol_rep.y.shape}.")
    # Apply the original scalar quantizer separately to x, y, z; keep the
    # original solver, sampling interval, full-window scaling, and no burn-in
    lorenz_sequence = np.column_stack([
        quantize_lorenz_x_values(coordinate) for coordinate in sol_rep.y[:NUM_CHANNELS]
    ])
    rep_inputs["Lorenz"] = {
        "sequence": lorenz_sequence if NUM_CHANNELS > 1 else lorenz_sequence[:, 0],
        "metadata": {
            "input_class": "lorenz", "rep_id": rep_id,
            "lorenz_initial_state": lorenz_initial_state.tolist(),
            "lorenz_initial_perturbation": lorenz_perturbation.tolist(),
            "lorenz_x0_rep": float(lorenz_initial_state[0]),
            "lorenz_y0_rep": float(lorenz_initial_state[1]),
            "lorenz_z0_rep": float(lorenz_initial_state[2]),
            "used_burn_in": False,
            "channels": [
                {"coordinate": coordinate_name,
                 "initial_value": float(lorenz_initial_state[k]),
                 "quantization_min": float(sol_rep.y[k].min()),
                 "quantization_max": float(sol_rep.y[k].max())}
                for k, coordinate_name in enumerate(("x", "y", "z")[:NUM_CHANNELS])
            ],
        },
    }

    # Validate token sequences before model inference
    for data_name in input_data_order_full:
        sequence = np.asarray(rep_inputs[str(data_name)]["sequence"], dtype=int)
        expected_shape = (TOTAL_SEQUENCE_STATES, NUM_CHANNELS) if NUM_CHANNELS > 1 else (TOTAL_SEQUENCE_STATES,)
        if sequence.shape != expected_shape:
            raise ValueError(
                f"{data_name}, rep_id={rep_id}: expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape}."
            )
        if not np.all((0 <= sequence) & (sequence <= 999)):
            raise ValueError(f"{data_name}, rep_id={rep_id}: quantized states must be in [0, 999].")

    ensemble_inputs[rep_id] = rep_inputs

if len(ensemble_inputs) != NUM_ENSEMBLE_RUNS:
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} ensemble runs, got {len(ensemble_inputs)}.")
if np.unique(constant_tokens, axis=0).shape[0] <= 1:
    raise ValueError("Expected sampled constant tokens to vary across ensemble replicates.")

# Store reproducible inputs and provenance alongside both raw and summarized results
ensemble_sequences = np.asarray([
    [ensemble_inputs[int(rep_id)][str(name)]["sequence"] for rep_id in run_ids]
    for name in input_names
], dtype=np.int64)
ensemble_metadata_json = np.asarray([
    [json.dumps(ensemble_inputs[int(rep_id)][str(name)]["metadata"], sort_keys=True)
     for rep_id in run_ids] for name in input_names
], dtype=str)
experiment_config_json = json.dumps({
    "suite": INPUT_SUITE, "num_channels": NUM_CHANNELS,
    "seed": SEED, "ensemble_seed": ENSEMBLE_SEED,
    "quantization": QUANTIZATION_CONVENTION,
    "constant_quantization": CONSTANT_QUANTIZATION_CONVENTION,
    "lorenz_initial_base": LORENZ_BASE_INITIAL_STATE.tolist(),
    "lorenz_initial_perturbation": "independent_u_minus_0.5_for_each_observed_coordinate",
    "lorenz_parameters": {"sigma": sigma, "rho": rho, "beta": beta},
    "lorenz_dt": lorenz_dt, "lorenz_t_span": list(t_span),
    "logistic_long_n": logistic_long_n, "logistic_settle_tol": logistic_settle_tol,
    "periodic_random_phase": periodic_random_phase,
    "drop_bos": DROP_BOS, "renormalize_after_drop": RENORMALIZE_AFTER_DROP,
    "head_weighting": HEAD_WEIGHTING, "add_special_tokens": TOKENIZATION_ADD_SPECIAL_TOKENS,
    "signal": "hidden_states[layer_idx]", "state_delimiter": STATE_DELIMITER,
}, sort_keys=True)
experiment_fingerprint = hashlib.sha256(
    experiment_config_json.encode() + ensemble_sequences.tobytes()
    + ensemble_metadata_json.tobytes()
).hexdigest()


def experiment_metadata():
    return {
        "input_suite": np.asarray(INPUT_SUITE),
        "input_labels": input_labels, "input_class_labels": input_class_labels,
        "num_channels": np.asarray(NUM_CHANNELS),
        "tokens_per_time_step": np.asarray(TOKENS_PER_TIME_STEP),
        "target_context_token_counts": TARGET_CONTEXT_TOKEN_COUNTS[active_context_indices],
        "quantization_convention": np.asarray(QUANTIZATION_CONVENTION),
        "constant_quantization_convention": np.asarray(CONSTANT_QUANTIZATION_CONVENTION),
        "experiment_config_json": np.asarray(experiment_config_json),
        "experiment_fingerprint": np.asarray(experiment_fingerprint),
        "initial_conditions": x0_samples[run_ids],
        "ensemble_sequences": ensemble_sequences,
        "ensemble_metadata_json": ensemble_metadata_json,
        "std_ddof": np.asarray(1),
    }


Load the tokenizer, validate prompt tokenization, and precompute context-dependent cutoffs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def _single_token_id(token_str):
    token_ids = tokenizer(token_str, add_special_tokens=False).input_ids
    if len(token_ids) != 1:
        raise ValueError(f"Expected {token_str!r} to tokenize as one token; got {token_ids}.")
    return int(token_ids[0])


COMMA_TOKEN_ID = _single_token_id(",")
SEMICOLON_TOKEN_ID = _single_token_id(";")
STATE_DELIMITER_TOKEN_ID = _single_token_id(STATE_DELIMITER)
# Require one token per state and comma for exact context sizing
NUMERIC_TOKEN_IDS = np.asarray([_single_token_id(f"{i:03d}") for i in range(1000)], dtype=np.int64)
if len(np.unique(NUMERIC_TOKEN_IDS)) != 1000:
    raise ValueError("Expected all 000..999 formatted states to map to distinct single tokens.")


def _first_token_value(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        if x.numel() == 0:
            return None
        return int(x.flatten()[0].item())
    arr = np.asarray(x).reshape(-1)
    if arr.size == 0:
        return None
    return int(arr[0])


def get_token_start_index(
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
):
    # Exclude an optional leading BOS from graph and signal nodes
    if not drop_bos:
        return 0

    first_id = _first_token_value(input_ids)
    if first_id is not None and bos_token_id is not None and first_id == int(bos_token_id):
        return 1

    first_mask = _first_token_value(special_tokens_mask)
    if first_mask == 1:
        return 1

    return 0


def _build_input_sequence_from_frac(sequence, frac, total_n=MAX_CONTEXT_STATES):
    # Preserve scalar comma serialization; vectors use commas within a state
    # and a semicolon after EVERY state, including the final state (6T tokens)
    sequence = np.asarray(sequence, dtype=int)
    if sequence.ndim not in (1, 2) or (sequence.ndim == 2 and sequence.shape[1] != NUM_CHANNELS):
        raise ValueError(f"Unexpected trajectory shape: {sequence.shape}.")
    if sequence.shape[0] < int(total_n):
        raise ValueError(f"Need at least {int(total_n)} states; got {sequence.shape[0]}.")
    if np.min(sequence[:int(total_n)]) < 0 or np.max(sequence[:int(total_n)]) > 999:
        raise ValueError("Prompt state tokens must be in [0, 999].")

    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * int(total_n))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = sequence[:ctx_len]
    if sequence.ndim == 1:
        input_sequence = ",".join(f"{x:03d}" for x in context_window) + ","
    else:
        input_sequence = "".join(
            ",".join(f"{x:03d}" for x in state) + ";" for state in context_window
        )
    return input_sequence, int(ctx_len)


def _post_bos_token_ids(input_ids, token_start):
    flat_ids = input_ids.reshape(-1)
    return flat_ids[int(token_start):]


def tokenize_and_validate_diagnostic_prompt(prompt, expected_post_bos_tokens=None):
    if not prompt.endswith(STATE_DELIMITER):
        raise ValueError(f"Diagnostic prompt must end with {STATE_DELIMITER!r}.")
    # Validate the FULL token stream against the intended numeric/punctuation
    # tokens, rather than assuming single-token pieces stay separate in context
    expected_ids = []
    for state in prompt[:-1].split(STATE_DELIMITER):
        channels = state.split(",") if NUM_CHANNELS > 1 else [state]
        if len(channels) != NUM_CHANNELS:
            raise ValueError(f"Expected {NUM_CHANNELS} channels per state.")
        for channel_idx, value in enumerate(channels):
            if len(value) != 3 or not value.isascii() or not value.isdigit():
                raise ValueError(f"Expected a three-digit integer, got {value!r}.")
            expected_ids.append(int(NUMERIC_TOKEN_IDS[int(value)]))
            expected_ids.append(
                COMMA_TOKEN_ID if channel_idx < NUM_CHANNELS - 1 else STATE_DELIMITER_TOKEN_ID
            )

    encoded = tokenizer(
        prompt,
        add_special_tokens=TOKENIZATION_ADD_SPECIAL_TOKENS,
        return_special_tokens_mask=True,
        return_tensors="pt",
    )
    input_ids = encoded.input_ids[0]
    special_tokens_mask = encoded.special_tokens_mask[0]
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=DROP_BOS,
    )
    post_bos_ids = _post_bos_token_ids(input_ids, token_start)
    seq_len = int(post_bos_ids.shape[0])
    if expected_post_bos_tokens is not None and seq_len != int(expected_post_bos_tokens):
        raise ValueError(
            f"Expected {int(expected_post_bos_tokens)} post-BOS content tokens, got {seq_len}."
        )
    if seq_len <= LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES + 1:
        raise ValueError(
            f"Need at least {LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES + 2} post-BOS tokens; got {seq_len}."
        )
    if post_bos_ids.tolist() != expected_ids:
        raise ValueError("Numeric values and delimiters must each remain individual tokens in context.")
    return {
        "encoded": encoded,
        "token_start": int(token_start),
        "seq_len": int(seq_len),
        "final_token_is_state_delimiter": True,
    }


reference_sequence = np.arange(MAX_CONTEXT_STATES * NUM_CHANNELS, dtype=int) % 1000
if NUM_CHANNELS > 1:
    reference_sequence = reference_sequence.reshape(MAX_CONTEXT_STATES, NUM_CHANNELS)
context_state_counts = np.zeros(len(context_fracs), dtype=np.int64)
context_token_counts = np.zeros(len(context_fracs), dtype=np.int64)
# Measure exact post-BOS lengths with a reference prompt
for context_idx, frac in enumerate(context_fracs):
    reference_prompt, ctx_len = _build_input_sequence_from_frac(
        reference_sequence,
        frac,
        total_n=MAX_CONTEXT_STATES,
    )
    prompt_info = tokenize_and_validate_diagnostic_prompt(reference_prompt)
    context_state_counts[context_idx] = int(ctx_len)
    context_token_counts[context_idx] = int(prompt_info["seq_len"])

expected_context_state_counts = np.maximum(
    1,
    np.rint(context_fracs * MAX_CONTEXT_STATES).astype(np.int64),
)
if not np.array_equal(context_state_counts, expected_context_state_counts):
    raise ValueError(
        f"Expected exact context state counts {expected_context_state_counts.tolist()}, "
        f"got {context_state_counts.tolist()}."
    )
if not np.array_equal(context_token_counts, TOKENS_PER_TIME_STEP * context_state_counts):
    raise ValueError("Expected exactly 2 * NUM_CHANNELS tokens per time step.")

k_cutoff_by_context = context_token_counts // 2

hfer_tail_mode_counts_by_context = np.zeros(
    (len(context_token_counts), len(HFER_TAIL_FRACTIONS)),
    dtype=np.int64,
)
hfer_tail_cutoff_by_context = np.zeros(
    (len(context_token_counts), len(HFER_TAIL_FRACTIONS)),
    dtype=np.int64,
)
# Precompute HFER high-frequency tail cutoffs
for context_idx, seq_len in enumerate(context_token_counts):
    seq_len = int(seq_len)
    for tail_idx, tail_fraction in enumerate(HFER_TAIL_FRACTIONS):
        tail_count = int(round(float(tail_fraction) * seq_len))
        tail_count = min(max(tail_count, 0), seq_len)
        cutoff = seq_len - tail_count
        hfer_tail_mode_counts_by_context[context_idx, tail_idx] = tail_count
        hfer_tail_cutoff_by_context[context_idx, tail_idx] = cutoff

if not np.all(np.diff(context_token_counts) > 0):
    raise ValueError(f"context_token_counts must be strictly increasing; got {context_token_counts.tolist()}.")
if int(context_token_counts[-1]) != TOKENS_PER_TIME_STEP * MAX_CONTEXT_STATES:
    raise ValueError("Full context token count disagrees with complete-state serialization.")
print(f"{INPUT_SUITE}: T={context_state_counts.tolist()}, N={context_token_counts.tolist()}")


Compute the graph-spectral diagnostics for one layer from attention and hidden-state tensors.

In [ ]:
def compute_all_layer_diagnostics_from_attention_and_hidden(
    attention_layer,
    hidden_state_layer,
    input_ids,
    layer_idx,
    expected_seq_len,
    k_cutoff=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
    renormalize_after_drop=True,
    head_weighting="mass",
    low_spectrum_num_nontrivial=LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES,
    hfer_tail_fractions=HFER_TAIL_FRACTIONS,
):
    # Remove the same leading BOS from attention and hidden states
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=drop_bos,
    )

    A = attention_layer[0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len != int(expected_seq_len):
        raise ValueError(f"Layer {layer_idx}: expected seq_len={expected_seq_len}, got {seq_len}.")
    if seq_len <= low_spectrum_num_nontrivial + 1:
        raise ValueError(
            f"Layer {layer_idx}: seq_len={seq_len} is too small for {low_spectrum_num_nontrivial} nontrivial eigenvalues."
        )

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    # Symmetrize heads and combine them into one weighted graph
    if k_cutoff is None:
        k_cutoff = seq_len // 2
    k_cutoff = int(k_cutoff)
    if k_cutoff < 0 or k_cutoff > seq_len:
        raise ValueError(f"Layer {layer_idx}: invalid k_cutoff={k_cutoff} for seq_len={seq_len}.")

    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'.")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    deg = Wbar.sum(dim=1)

    L = torch.diag(deg) - Wbar
    _, U = torch.linalg.eigh(L)

    # Project hidden states onto graph Fourier modes
    X = hidden_state_layer[0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Layer {layer_idx}: hidden-state length mismatch after BOS drop; X={X.shape[0]}, graph={seq_len}."
        )

    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total_energy = energy.sum()
    if total_energy <= 0 or not torch.isfinite(total_energy):
        raise ValueError(f"Layer {layer_idx}: nonpositive or nonfinite total signal energy: {total_energy}.")
    p = energy / total_energy
    positive_p = p[p > 0]
    spectral_entropy = -torch.sum(positive_p * torch.log(positive_p))
    effective_active_modes = torch.exp(spectral_entropy)

    # Store HFER for the top 50% and narrower high-frequency tails
    default_hfer = float(p[k_cutoff:].sum().detach().cpu().item())
    hfer_tail_energies = []
    hfer_tail_cutoffs = []
    hfer_tail_mode_counts = []
    for tail_fraction in hfer_tail_fractions:
        tail_fraction = float(tail_fraction)
        if tail_fraction < 0.0 or tail_fraction > 1.0:
            raise ValueError("HFER tail fractions must be in [0, 1].")
        tail_count = int(round(tail_fraction * seq_len))
        tail_count = min(max(tail_count, 0), seq_len)
        cutoff = seq_len - tail_count
        hfer_tail_energies.append(float(p[cutoff:seq_len].sum().detach().cpu().item()))
        hfer_tail_cutoffs.append(int(cutoff))
        hfer_tail_mode_counts.append(int(tail_count))

    if not np.isclose(default_hfer, hfer_tail_energies[0], rtol=1e-6, atol=1e-7):
        raise ValueError("Top-50% HFER tail energy must match default k_cutoff HFER.")
    hfer = float(hfer_tail_energies[0])

    # Include lambda_2 (the normalized Fiedler value) in the low-end spectrum
    inv_sqrt_deg = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    Wbar_norm = inv_sqrt_deg[:, None] * Wbar * inv_sqrt_deg[None, :]
    L_sym = torch.eye(seq_len, device=Wbar.device, dtype=Wbar.dtype) - Wbar_norm
    eigvals_sym = torch.linalg.eigvalsh(L_sym)
    low_spectrum_eigvals = eigvals_sym[1:1 + low_spectrum_num_nontrivial]
    if low_spectrum_eigvals.numel() != low_spectrum_num_nontrivial:
        raise ValueError(
            f"Layer {layer_idx}: expected {low_spectrum_num_nontrivial} low-end eigenvalues, "
            f"got {low_spectrum_eigvals.numel()}."
        )

    result = {
        "effective_active_modes": float(effective_active_modes.detach().cpu().item()),
        "normalized_fiedler_value": float(eigvals_sym[1].detach().cpu().item()),
        "hfer": hfer,
        "hfer_tail_energies": np.asarray(hfer_tail_energies, dtype=np.float64),
        "hfer_tail_cutoffs": np.asarray(hfer_tail_cutoffs, dtype=np.int64),
        "hfer_tail_mode_counts": np.asarray(hfer_tail_mode_counts, dtype=np.int64),
        "low_spectrum_eigvals": low_spectrum_eigvals.detach().cpu().numpy().astype(np.float64),
        "seq_len": int(seq_len),
        "k_cutoff": int(k_cutoff),
        "token_start": int(token_start),
        "attention_layer_index": int(layer_idx),
        "signal_layer_index": int(layer_idx),
    }

    del A, W_heads, Wbar, deg, L, U, X, X_hat, energy, p, inv_sqrt_deg, Wbar_norm, L_sym, eigvals_sym
    return result



Create, validate, load, and save checkpoint arrays for resumable sweeps.

In [ ]:
def _expected_result_shapes():
    base_shape = (len(input_names), len(run_ids), len(context_fracs))
    low_shape = base_shape + (LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES,)
    hfer_tail_shape = base_shape + (len(HFER_TAIL_FRACTIONS),)
    return base_shape, low_shape, hfer_tail_shape


def _initialize_result_arrays():
    # Retain per-run values for summary recomputation
    base_shape, low_shape, hfer_tail_shape = _expected_result_shapes()
    return {
        "active_modes_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "fiedler_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "hfer_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "fiedler_lsym_low_spectrum_layeravg_per_run": np.full(low_shape, np.nan, dtype=np.float64),
        "hfer_tail_energy_layeravg_per_run": np.full(hfer_tail_shape, np.nan, dtype=np.float64),
        "completed_mask": np.zeros(base_shape, dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def _validate_checkpoint_compatibility(data, num_layers):
    # Accept only checkpoints matching the model, sweep, inputs, and schema
    if _npz_scalar_to_str(data, "experiment_fingerprint") != experiment_fingerprint:
        raise ValueError("Checkpoint input/configuration fingerprint mismatch.")
    if _npz_scalar_to_str(data, "checkpoint_schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Checkpoint schema version mismatch.")
    if _npz_scalar_to_str(data, "npz_schema_version") != NPZ_SCHEMA_VERSION:
        raise ValueError("Checkpoint final schema version mismatch.")
    if _npz_scalar_to_str(data, "model_name") != MODEL_NAME:
        raise ValueError("Checkpoint model_name mismatch.")
    if int(np.asarray(data["num_layers"]).item()) != int(num_layers):
        raise ValueError("Checkpoint num_layers mismatch.")
    if int(np.asarray(data["low_spectrum_num_nontrivial"]).item()) != LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES:
        raise ValueError("Checkpoint low-spectrum K mismatch.")
    if not np.array_equal(data["input_names"].astype(str), input_names.astype(str)):
        raise ValueError("Checkpoint input_names mismatch.")
    if not np.array_equal(data["run_ids"].astype(np.int64), run_ids.astype(np.int64)):
        raise ValueError("Checkpoint run_ids mismatch.")
    if not np.allclose(data["context_fracs"].astype(np.float64), context_fracs.astype(np.float64)):
        raise ValueError("Checkpoint context_fracs mismatch.")
    if int(np.asarray(data["max_context_states"]).item()) != MAX_CONTEXT_STATES:
        raise ValueError("Checkpoint max_context_states mismatch.")
    if int(np.asarray(data["total_sequence_states"]).item()) != TOTAL_SEQUENCE_STATES:
        raise ValueError("Checkpoint total_sequence_states mismatch.")
    if not np.array_equal(data["context_state_counts"].astype(np.int64), context_state_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_state_counts mismatch.")
    if not np.array_equal(data["context_token_counts"].astype(np.int64), context_token_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_token_counts mismatch.")
    if not np.array_equal(data["k_cutoff_by_context"].astype(np.int64), k_cutoff_by_context.astype(np.int64)):
        raise ValueError("Checkpoint k_cutoff_by_context mismatch.")
    if not np.allclose(data["hfer_tail_fractions"].astype(np.float64), HFER_TAIL_FRACTIONS.astype(np.float64)):
        raise ValueError("Checkpoint HFER tail fractions mismatch.")
    if not np.array_equal(data["hfer_tail_labels"].astype(str), HFER_TAIL_LABELS.astype(str)):
        raise ValueError("Checkpoint HFER tail labels mismatch.")

    expected_base_shape, expected_low_shape, expected_hfer_tail_shape = _expected_result_shapes()
    shape_checks = {
        "active_modes_layeravg_per_run": expected_base_shape,
        "fiedler_layeravg_per_run": expected_base_shape,
        "hfer_layeravg_per_run": expected_base_shape,
        "fiedler_lsym_low_spectrum_layeravg_per_run": expected_low_shape,
        "hfer_tail_energy_layeravg_per_run": expected_hfer_tail_shape,
        "completed_mask": expected_base_shape,
    }
    for key, expected_shape in shape_checks.items():
        if data[key].shape != expected_shape:
            raise ValueError(f"Checkpoint {key} shape {data[key].shape}, expected {expected_shape}.")


def load_or_initialize_checkpoint(num_layers):
    if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
        with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
            _validate_checkpoint_compatibility(data, num_layers=num_layers)
            arrays = {
                "active_modes_layeravg_per_run": data["active_modes_layeravg_per_run"].astype(np.float64),
                "fiedler_layeravg_per_run": data["fiedler_layeravg_per_run"].astype(np.float64),
                "hfer_layeravg_per_run": data["hfer_layeravg_per_run"].astype(np.float64),
                "fiedler_lsym_low_spectrum_layeravg_per_run": data[
                    "fiedler_lsym_low_spectrum_layeravg_per_run"
                ].astype(np.float64),
                "hfer_tail_energy_layeravg_per_run": data[
                    "hfer_tail_energy_layeravg_per_run"
                ].astype(np.float64),
                "completed_mask": data["completed_mask"].astype(bool),
            }
        return arrays

    return _initialize_result_arrays()


def save_checkpoint(arrays, num_layers):
    # Save progress with compatibility metadata
    # Atomic replacement preserves the previous checkpoint if saving is interrupted
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary_path,
        **experiment_metadata(),
        input_names=input_names,
        context_fracs=context_fracs,
        context_state_counts=context_state_counts,
        context_token_counts=context_token_counts,
        k_cutoff_by_context=k_cutoff_by_context,
        run_ids=run_ids,
        model_name=np.asarray(MODEL_NAME),
        max_context_states=np.asarray(MAX_CONTEXT_STATES, dtype=np.int64),
        total_sequence_states=np.asarray(TOTAL_SEQUENCE_STATES, dtype=np.int64),
        num_layers=np.asarray(num_layers, dtype=np.int64),
        low_spectrum_num_nontrivial=np.asarray(LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES, dtype=np.int64),
        hfer_tail_fractions=HFER_TAIL_FRACTIONS,
        hfer_tail_labels=HFER_TAIL_LABELS,
        npz_schema_version=np.asarray(NPZ_SCHEMA_VERSION),
        checkpoint_schema_version=np.asarray(CHECKPOINT_SCHEMA_VERSION),
        active_modes_layeravg_per_run=arrays["active_modes_layeravg_per_run"],
        fiedler_layeravg_per_run=arrays["fiedler_layeravg_per_run"],
        hfer_layeravg_per_run=arrays["hfer_layeravg_per_run"],
        fiedler_lsym_low_spectrum_layeravg_per_run=arrays[
            "fiedler_lsym_low_spectrum_layeravg_per_run"
        ],
        hfer_tail_energy_layeravg_per_run=arrays["hfer_tail_energy_layeravg_per_run"],
        completed_mask=arrays["completed_mask"],
    )
    temporary_path.replace(CHECKPOINT_PATH)


Load the model with eager attention and capture layer/device metadata.

In [ ]:
# Use eager attention with automatic model placement
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

num_layers = int(model_attn.config.num_hidden_layers)
if num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {num_layers}.")

layer_indices = np.arange(num_layers, dtype=np.int64)
device = next(model_attn.parameters()).device


Run or resume the full sweep, averaging diagnostics across layers and checkpointing progress.

In [ ]:
arrays = load_or_initialize_checkpoint(num_layers=num_layers)
completed_since_checkpoint = 0

# Task grid: input family x realization x context length
tasks = [
    (input_axis_idx, str(data_name), run_axis_idx, int(rep_id), context_idx)
    for input_axis_idx, data_name in enumerate(input_names)
    for run_axis_idx, rep_id in enumerate(run_ids)
    for context_idx in range(len(context_fracs))
]

checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_TASKS))
initial_completed_total = int(arrays["completed_mask"].sum())
next_checkpoint_completed = (
    (initial_completed_total // checkpoint_every) + 1
) * checkpoint_every

for input_axis_idx, data_name, run_axis_idx, rep_id, context_idx in tqdm(
    tasks,
    desc="All diagnostics layer-averaged vs context",
    miniters=checkpoint_every,
):
    # Skip tasks saved in a compatible checkpoint
    if arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx]:
        continue

    sequence = np.asarray(ensemble_inputs[int(rep_id)][data_name]["sequence"], dtype=int)
    prompt, _ = _build_input_sequence_from_frac(
        sequence,
        context_fracs[context_idx],
        total_n=MAX_CONTEXT_STATES,
    )
    prompt_info = tokenize_and_validate_diagnostic_prompt(
        prompt,
        expected_post_bos_tokens=int(context_token_counts[context_idx]),
    )
    encoded = prompt_info["encoded"]
    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask", "position_ids", "token_type_ids"}
    }
    input_ids_for_diagnostics = model_inputs["input_ids"][0]
    special_tokens_mask = (
        encoded["special_tokens_mask"][0].to(device)
        if "special_tokens_mask" in encoded
        else None
    )
    expected_seq_len = int(context_token_counts[context_idx])
    expected_k_cutoff = int(k_cutoff_by_context[context_idx])

    # Request attention and hidden states for each layer
    with torch.no_grad():
        outputs = model_attn(
            **model_inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,  # No generation; avoid unused key/value tensors.
        )

    if outputs.attentions is None or outputs.hidden_states is None:
        raise ValueError("Model outputs must include attentions and hidden_states.")
    if len(outputs.attentions) != num_layers:
        raise ValueError(f"Expected {num_layers} attention tensors, got {len(outputs.attentions)}.")
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError(f"Expected {num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}.")

    active_modes_sum = 0.0
    fiedler_sum = 0.0
    hfer_sum = 0.0
    low_spectrum_sum = np.zeros(LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES, dtype=np.float64)
    hfer_tail_energy_sum = np.zeros(len(HFER_TAIL_FRACTIONS), dtype=np.float64)

    # Accumulate and average each diagnostic across layers
    for layer_idx in range(num_layers):
        bundle = compute_all_layer_diagnostics_from_attention_and_hidden(
            outputs.attentions[layer_idx],
            outputs.hidden_states[layer_idx],
            input_ids=input_ids_for_diagnostics,
            layer_idx=layer_idx,
            expected_seq_len=expected_seq_len,
            k_cutoff=expected_k_cutoff,
            bos_token_id=tokenizer.bos_token_id,
            special_tokens_mask=special_tokens_mask,
            drop_bos=DROP_BOS,
            renormalize_after_drop=RENORMALIZE_AFTER_DROP,
            head_weighting=HEAD_WEIGHTING,
            low_spectrum_num_nontrivial=LOW_SPECTRUM_NUM_NONTRIVIAL_EIGENVALUES,
            hfer_tail_fractions=HFER_TAIL_FRACTIONS,
        )
        active_modes_sum += float(bundle["effective_active_modes"])
        fiedler_sum += float(bundle["normalized_fiedler_value"])
        hfer_sum += float(bundle["hfer"])
        low_spectrum_sum += bundle["low_spectrum_eigvals"].astype(np.float64)
        hfer_tail_energy_sum += bundle["hfer_tail_energies"].astype(np.float64)
        del bundle

    active_modes_layeravg = active_modes_sum / num_layers
    fiedler_layeravg = fiedler_sum / num_layers
    hfer_layeravg = hfer_sum / num_layers
    low_spectrum_layeravg = low_spectrum_sum / num_layers
    hfer_tail_energy_layeravg = hfer_tail_energy_sum / num_layers

    arrays["active_modes_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = active_modes_layeravg
    arrays["fiedler_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = fiedler_layeravg
    arrays["hfer_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = hfer_layeravg
    arrays["fiedler_lsym_low_spectrum_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx, :] = low_spectrum_layeravg
    arrays["hfer_tail_energy_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx, :] = hfer_tail_energy_layeravg
    arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx] = True
    completed_since_checkpoint += 1

    # Release large tensors between tasks
    del outputs, model_inputs, encoded, prompt_info, input_ids_for_diagnostics, special_tokens_mask
    del active_modes_sum, fiedler_sum, hfer_sum, low_spectrum_sum, hfer_tail_energy_sum, hfer_tail_energy_layeravg
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    completed_total = int(arrays["completed_mask"].sum())
    if completed_total >= next_checkpoint_completed:
        save_checkpoint(arrays, num_layers=num_layers)
        completed_since_checkpoint = 0
        next_checkpoint_completed = (
            (completed_total // checkpoint_every) + 1
        ) * checkpoint_every

if completed_since_checkpoint > 0 or not CHECKPOINT_PATH.exists():
    save_checkpoint(arrays, num_layers=num_layers)


Validate completed arrays, summarize over runs, and save the visualization cache.

In [ ]:
def _summarize_over_runs(per_run):
    mean = np.mean(per_run, axis=1)
    if per_run.shape[1] > 1:
        std = np.std(per_run, axis=1, ddof=1)
    else:
        std = np.zeros_like(mean)
    sem = std / np.sqrt(per_run.shape[1])
    return mean, std, sem


def _validate_final_arrays():
    # Reject incomplete or invalid arrays before saving
    if not np.all(arrays["completed_mask"]):
        bad = np.argwhere(~arrays["completed_mask"])[:10]
        raise ValueError(f"Not all tasks are complete; first incomplete indices: {bad}.")

    base_shape, low_shape, hfer_tail_shape = _expected_result_shapes()
    expected_shapes = {
        "active_modes_layeravg_per_run": base_shape,
        "fiedler_layeravg_per_run": base_shape,
        "hfer_layeravg_per_run": base_shape,
        "fiedler_lsym_low_spectrum_layeravg_per_run": low_shape,
        "hfer_tail_energy_layeravg_per_run": hfer_tail_shape,
    }
    for key, expected_shape in expected_shapes.items():
        if arrays[key].shape != expected_shape:
            raise ValueError(f"{key} shape {arrays[key].shape}, expected {expected_shape}.")

    active = arrays["active_modes_layeravg_per_run"]
    fiedler = arrays["fiedler_layeravg_per_run"]
    hfer = arrays["hfer_layeravg_per_run"]
    low = arrays["fiedler_lsym_low_spectrum_layeravg_per_run"]
    hfer_tail = arrays["hfer_tail_energy_layeravg_per_run"]

    for key, value in {
        "active_modes_layeravg_per_run": active,
        "fiedler_layeravg_per_run": fiedler,
        "hfer_layeravg_per_run": hfer,
        "fiedler_lsym_low_spectrum_layeravg_per_run": low,
        "hfer_tail_energy_layeravg_per_run": hfer_tail,
    }.items():
        if not np.isfinite(value).all():
            bad = np.argwhere(~np.isfinite(value))[:10]
            raise ValueError(f"{key} contains nonfinite values; first bad indices: {bad}.")

    if not np.all(active > 0):
        raise ValueError("Active modes must be positive.")
    if not np.all((hfer >= -1e-6) & (hfer <= 1.0 + 1e-6)):
        raise ValueError("HFER values must be in [0, 1] up to tiny numerical tolerance.")
    if not np.all((hfer_tail >= -1e-6) & (hfer_tail <= 1.0 + 1e-6)):
        raise ValueError("HFER tail-energy values must be in [0, 1] up to tiny numerical tolerance.")
    if not np.all((fiedler >= -1e-4) & (fiedler <= 2.0 + 1e-4)):
        raise ValueError("Normalized Fiedler values must be in [0, 2] up to numerical tolerance.")
    if not np.all((low >= -1e-4) & (low <= 2.0 + 1e-4)):
        raise ValueError("Low-spectrum normalized-Laplacian eigenvalues must be in [0, 2] up to numerical tolerance.")
    if not np.all(np.diff(low, axis=-1) >= -1e-5):
        bad = np.argwhere(np.diff(low, axis=-1) < -1e-5)[:10]
        raise ValueError(f"Each low-spectrum vector must be nondecreasing; first bad indices: {bad}.")
    if not np.allclose(fiedler, low[..., 0], rtol=1e-5, atol=1e-6):
        raise ValueError("fiedler_layeravg_per_run must equal low-spectrum lambda_2 component.")
    if not np.allclose(hfer_tail[..., 0], hfer, rtol=1e-5, atol=1e-6):
        raise ValueError("hfer_tail_energy_layeravg_per_run[..., 0] must match default HFER.")
    if not np.all(hfer_tail[..., :-1] + 1e-6 >= hfer_tail[..., 1:]):
        raise ValueError("HFER tail energies must decrease as stored tail fractions shrink.")


_validate_final_arrays()

# Summarize across realizations
active_modes_mean, active_modes_std, active_modes_sem = _summarize_over_runs(
    arrays["active_modes_layeravg_per_run"]
)
fiedler_mean, fiedler_std, fiedler_sem = _summarize_over_runs(
    arrays["fiedler_layeravg_per_run"]
)
hfer_mean, hfer_std, hfer_sem = _summarize_over_runs(
    arrays["hfer_layeravg_per_run"]
)
hfer_tail_energy_mean, hfer_tail_energy_std, hfer_tail_energy_sem = _summarize_over_runs(
    arrays["hfer_tail_energy_layeravg_per_run"]
)
(
    fiedler_lsym_low_spectrum_mean,
    fiedler_lsym_low_spectrum_std,
    fiedler_lsym_low_spectrum_sem,
) = _summarize_over_runs(arrays["fiedler_lsym_low_spectrum_layeravg_per_run"])

# Save arrays for visualization
np.savez_compressed(
    OUTPUT_PATH,
    **experiment_metadata(),
    **arrays,  # Raw per-realization layer averages and completion mask.
    npz_schema_version=np.asarray(NPZ_SCHEMA_VERSION),
    context_state_counts=context_state_counts,  # T: observed vector-valued time steps.
    context_fracs=context_fracs,
    max_context_states=np.asarray(MAX_CONTEXT_STATES),
    total_sequence_states=np.asarray(TOTAL_SEQUENCE_STATES),
    input_names=input_names,
    context_token_counts=context_token_counts,
    run_ids=run_ids,
    num_layers=np.asarray(num_layers, dtype=np.int64),
    num_ensemble_runs=np.asarray(len(run_ids), dtype=np.int64),
    model_name=np.asarray(MODEL_NAME),
    active_modes_mean=active_modes_mean,
    active_modes_std=active_modes_std,
    fiedler_mean=fiedler_mean,
    fiedler_std=fiedler_std,
    hfer_mean=hfer_mean,
    hfer_std=hfer_std,
    hfer_tail_fractions=HFER_TAIL_FRACTIONS,
    hfer_tail_energy_mean=hfer_tail_energy_mean,
    hfer_tail_energy_std=hfer_tail_energy_std,
    fiedler_lsym_low_spectrum_mean=fiedler_lsym_low_spectrum_mean,
    fiedler_lsym_low_spectrum_std=fiedler_lsym_low_spectrum_std,
    fiedler_lsym_low_spectrum_lambda_indices=LOW_SPECTRUM_LAMBDA_INDICES,
)

print(f"Saved raw per-realization results and mean/sample-std summaries to {OUTPUT_PATH}")
